# Milestone 2: detector, extraction, PAM classifier, and evaluation

Run this notebook after pushing the repository to a public GitHub URL. It installs pinned direct dependencies, checks the fixed YOLOv5 revision and interfaces, runs tests and a CPU smoke path, repeats the approved visibility checks, generates synthetic data, trains the detector and PAM CNN, evaluates both test distributions, and saves the full run artifacts to Google Drive.

The notebook uses synthetic signals only. It does not fetch DeepSig or the Zenodo companion data.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import datetime
import subprocess
import sys

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(f"Pinned environment supports Python 3.10 through 3.12; this runtime is {sys.version.split()[0]}")

REPO_URL = input('Paste the public HTTPS GitHub URL for this project (no credentials): ').strip()
if not REPO_URL.startswith('https://github.com/'):
    raise ValueError('Use the public https://github.com/<owner>/<repo>.git URL.')

PROJECT_DIR = Path('/content/wideband-signal-recognition')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    current = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if current.removesuffix('.git') != REPO_URL.removesuffix('.git'):
        raise RuntimeError(f'Existing project checkout has a different origin: {current}')

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_OUT = Path('/content/drive/MyDrive/wideband_signal_recognition') / f'milestone2_{RUN_ID}'
LOCAL_WORK = Path('/content/wideband_m2_work') / RUN_ID
DRIVE_OUT.mkdir(parents=True, exist_ok=False)
(DRIVE_OUT / 'results').mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_DIR)
print('Drive output:', DRIVE_OUT)


## Install pinned dependencies

The project pins its direct dependencies and fetches YOLOv5 by a full commit hash. The exact YOLO source license, train and detect options, model file, and AutoAnchor API are checked before training.


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    '--extra-index-url', 'https://pypi.org/simple',
    '-r', str(PROJECT_DIR / 'requirements-colab.txt'),
], check=True)

import torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if not torch.__version__.startswith('2.5.1'):
    raise RuntimeError(f'Expected pinned PyTorch 2.5.1, got {torch.__version__}')
if torch.version.cuda != '12.1':
    raise RuntimeError(f'Expected CUDA 12.1 build, got {torch.version.cuda}')
if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU runtime in Colab before running the full milestone.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
YOLOV5_COMMIT = 'f9578796b8a64889aa9e66dc06914d19f40d338e'
YOLO_DIR = Path('/content/yolov5_pinned_f9578796')
if not YOLO_DIR.exists():
    YOLO_DIR.mkdir(parents=True, exist_ok=False)
    subprocess.run(['git', 'init', str(YOLO_DIR)], check=True)
    subprocess.run(['git', '-C', str(YOLO_DIR), 'remote', 'add', 'origin', 'https://github.com/ultralytics/yolov5.git'], check=True)
    subprocess.run(['git', '-C', str(YOLO_DIR), 'fetch', '--depth', '1', 'origin', YOLOV5_COMMIT], check=True)
    subprocess.run(['git', '-C', str(YOLO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual = subprocess.check_output(['git', '-C', str(YOLO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != YOLOV5_COMMIT:
    raise RuntimeError(f'YOLOv5 pin mismatch: expected {YOLOV5_COMMIT}, got {actual}')
print('Verified YOLOv5 commit:', actual)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT_DIR)], check=True)


## Run tests and the small CPU path

The smoke check exercises generation, spectrogram rendering, the energy geometry baseline, filtering/framing, and the PAM CNN. It is a plumbing check, not a performance result. Test logs are saved to Drive.


In [ ]:
import os

def run_logged(command, log_path, cwd):
    print('+', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)), cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    with Path(log_path).open('w', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

os.chdir(PROJECT_DIR)
run_logged([sys.executable, '-m', 'pytest', '-q'], DRIVE_OUT / 'results' / 'pytest.log', PROJECT_DIR)
run_logged([sys.executable, 'scripts/smoke_test.py'], DRIVE_OUT / 'results' / 'cpu_smoke.log', PROJECT_DIR)


## Confirm small-box visibility before training

The worst-seed image and JSON report are saved to Drive. If either check fails, the notebook stops before model training so the minimum bandwidth or duration can be reviewed.

In [ ]:
minus5_dir = DRIVE_OUT / 'visibility' / 'minus5db'
ten_db_dir = DRIVE_OUT / 'visibility' / 'ten_db_seed16'
run_logged([
    sys.executable, 'scripts/visibility_check.py', '--seeds', '50', '--snr-db', '-5',
    '--output-dir', minus5_dir,
], DRIVE_OUT / 'results' / 'visibility_minus5db.log', PROJECT_DIR)
run_logged([
    sys.executable, 'scripts/visibility_check.py', '--seeds', '1', '--seed-start', '16',
    '--snr-db', '10', '--output-dir', ten_db_dir,
], DRIVE_OUT / 'results' / 'visibility_ten_db.log', PROJECT_DIR)

import json
from IPython.display import Image as DisplayImage, display
for label, folder in [('minus five dB', minus5_dir), ('ten dB comparison', ten_db_dir)]:
    report = json.loads((folder / 'visibility_summary.json').read_text(encoding='utf-8'))
    print(f"{label}: passed={report['passed']}, worst local contrast SD={report['worst_local_contrast_sd_above_local_median']:.3f}")
    if not report['passed']:
        raise RuntimeError(f"Visibility check failed at {label}; review the saved report before training.")
display(DisplayImage(filename=str(minus5_dir / 'worst_seed_spectrogram.png')))


## Generate, train, and evaluate

The run regenerates I/Q from saved seeds for PAM extraction. It writes spectrograms, YOLO labels, and generator metadata to Drive before training; full I/Q arrays are not archived. Validation alone selects confidence thresholds, while both test sets stay frozen.


In [ ]:
run_logged([
    sys.executable, 'scripts/run_milestone2.py',
    '--config', PROJECT_DIR / 'configs' / 'milestone2.yaml',
    '--yolo-dir', YOLO_DIR,
    '--out-dir', DRIVE_OUT,
    '--data-work', LOCAL_WORK,
], DRIVE_OUT / 'results' / 'milestone2.log', PROJECT_DIR)


## Final summary to paste back

Paste only the printed summary text. Full metrics, confidence intervals, per-class tables, confusion matrices, plots, logs, and checkpoints remain in the Drive output folder.


In [ ]:
summary_path = DRIVE_OUT / 'results' / 'summary.txt'
if not summary_path.exists():
    raise FileNotFoundError(f'Run did not finish; inspect saved logs and partial outputs under {DRIVE_OUT}')
print(summary_path.read_text(encoding='utf-8'))
print('Drive output folder:', DRIVE_OUT)
